In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib

import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib

# Download data from pinecone


### Initialize Pinecone


In [ ]:
%pip install -q pinecone-client[grpc]

from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
from dotenv import load_dotenv

load_dotenv()

pc = Pinecone()

In [ ]:
INDEX_NAME = "ai-news-index"
index = pc.Index(INDEX_NAME)

### Initialize mongodb


In [ ]:
from dotenv import load_dotenv
import os
import pymongo


load_dotenv()

mongo_client = pymongo.MongoClient(os.getenv("MONGODB_URI"))

db = mongo_client["blogdb"]

collection = db["ai_news"]

collection.find_one()

### Get IDs


In [ ]:
all_ids = [str(x["_id"]) for x in collection.find({}, {"id": 1})]

### Download data from pinecone


In [ ]:
import time
from tqdm import tqdm


def fetch_all_vectors(index, all_ids: list[str], batch_size: int = 1000):
    all_data = []
    total_batches = (len(all_ids) + batch_size - 1) // batch_size  # Round up division

    start_time = time.time()

    try:
        for i in tqdm(
            range(0, len(all_ids), batch_size),
            total=total_batches,
            desc="Fetching vectors",
        ):
            batch_ids = all_ids[i : i + batch_size]
            batch_data = index.fetch(ids=batch_ids)
            all_data.extend(batch_data["vectors"].values())

    except KeyboardInterrupt:
        print("\nOperation interrupted by user. Returning partial results.")
    finally:
        end_time = time.time()
        print(f"Total time taken: {end_time - start_time:.2f} seconds")
        print(f"Fetched {len(all_data)} vectors out of {len(all_ids)} total IDs")

    return all_data


all_vectors = fetch_all_vectors(index, all_ids)

In [ ]:
from datetime import datetime

data_dict = [
    {
        "id": x["id"],
        **x["metadata"],
        "embedding": x["values"],
    }
    for x in all_vectors
]

# convert found_at and date from unix timestaamp to datetime


for x in data_dict:
    x["found_at"] = datetime.fromtimestamp(x["found_at"])
    x["date"] = datetime.fromtimestamp(x["date"])


data_dict[:2]

### Convert to dataframe


In [ ]:
df = pd.DataFrame(data_dict)

df.head(2)

# Clustering


In [ ]:
%pip install -qU hdbscan plotly nbformat langchain langchain_community langchain_core langchain_anthropic 

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.prompts import PromptTemplate
from langchain.schema import StrOutputParser
from textwrap import dedent


def generate_cluster_title(article_titles: list[str]) -> str:
    llm = ChatAnthropic(model_name="claude-3-haiku-20240307")  # type:ignore

    prompt = dedent("""Given the following news article titles, generate a single 3-4 words title that summarizes what the cluster of articles is about:

                                <titles>
                                {titles}
                                </titles>

                                Summary title:""")

    prompt_template = PromptTemplate.from_template(prompt)

    chain = prompt_template | llm | StrOutputParser()

    titles = "\n".join(f"- {title}" for title in article_titles[:5])

    return chain.invoke({"titles": titles})


generate_cluster_title(
    [
        "Etched is building an AI chip that only runs one type of model",
        "Transformer model chipmaker Etched.ai raises $120M to challenge Nvidia's market dominance",
        "AI chip startup Etched raises $120 million to expand supply",
        "Etched raises $120M in challenge to Nvidia in AI with transformer chips",
        "NVIDIA-Backed Astrocade Raises $12M For A Promising AI Game Creation Platform",
    ]
)

In [ ]:
import pandas as pd
import hdbscan
from collections import defaultdict
from sklearn.metrics.pairwise import euclidean_distances
import plotly.express as px
import textwrap


def wrap_text(text, width=50):
    return "<br>".join(textwrap.wrap(text, width=width))


def get_cluster_center(points):
    return np.mean(points, axis=0)


def get_closest_points(points, center, n=5):
    distances = euclidean_distances([center], points)[0]
    closest_indices = np.argsort(distances)[:n]
    return closest_indices


matrix = np.array(df.embedding.tolist())

# Perform clustering on the original high-dimensional data
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=5)
cluster_labels = clusterer.fit_predict(matrix)

# Reduce dimensionality to 2D for visualization
tsne = TSNE(
    n_components=2,
    perplexity=15,
    random_state=42,
    init="random",
    learning_rate=150,
)
vis_dims = tsne.fit_transform(matrix)

# Create a new DataFrame with the t-SNE results, titles, wrapped bodies, and cluster labels
tsne_df = pd.DataFrame(vis_dims, columns=["tsne_1", "tsne_2"])
tsne_df["title"] = df["title"]
tsne_df["wrapped_body"] = df["body"].apply(wrap_text)
tsne_df["cluster"] = cluster_labels

# Group points and titles by cluster
cluster_points = defaultdict(list)
cluster_titles = defaultdict(list)
for cluster, point, title in zip(tsne_df["cluster"], matrix, tsne_df["title"]):
    cluster_points[cluster].append(point)
    cluster_titles[cluster].append(title)

# Generate titles for each cluster using the 5 closest articles to the center
cluster_names = {-1: "Noise"}  # Keep 'Noise' for cluster -1
for cluster, points in cluster_points.items():
    if cluster != -1:
        center = get_cluster_center(points)
        closest_indices = get_closest_points(points, center)
        closest_titles = [cluster_titles[cluster][i] for i in closest_indices]
        cluster_names[cluster] = generate_cluster_title(closest_titles)

# Apply the new cluster names
tsne_df["cluster_name"] = tsne_df["cluster"].map(cluster_names)

# Update the 'cluster' column with the new names
tsne_df["cluster"] = tsne_df["cluster_name"]

# Drop the temporary 'cluster_name' column
tsne_df = tsne_df.drop("cluster_name", axis=1)

# Create an interactive scatter plot using Plotly
fig = px.scatter(
    tsne_df,
    x="tsne_1",
    y="tsne_2",
    color="cluster",
    hover_data=["title", "wrapped_body", "cluster"],
    title="2D t-SNE projection of news articles with HDBSCAN clustering",
    labels={"tsne_1": "t-SNE feature 1", "tsne_2": "t-SNE feature 2"},
    color_discrete_sequence=px.colors.qualitative.Plotly,
)

# Update the hover template to show title, wrapped body, and cluster name
fig.update_traces(
    hovertemplate="<b>Title:</b> %{customdata[0]}<br><br>"
    "<b>Body:</b> %{customdata[1]}<br><br>"
    "<b>Cluster:</b> %{customdata[2]}"
)

# Adjust the hover mode to show the full text
fig.update_layout(hoverdistance=100, hovermode="closest")

# Show the plot
fig.show()

# Print the number of clusters
n_clusters = len(tsne_df["cluster"].unique()) - 1  # Exclude 'Noise' cluster
print(f"Number of clusters: {n_clusters}")

# Print cluster names
print("Cluster names:")
for cluster, name in cluster_names.items():
    if cluster != -1:
        print(f"Cluster {cluster}: {name}")

# Print articles in the 'Noise' cluster
noise_articles = tsne_df[tsne_df["cluster"] == "Noise"]
print(f"\nNumber of articles in 'Noise' cluster: {len(noise_articles)}")
print("Sample of articles in 'Noise' cluster:")
print(noise_articles["title"].head(10).tolist())